# Finding +EV Bets with SportsQuant

**How to identify profitable betting opportunities**

In sports betting, the single most important concept is **Expected Value (EV)**. A bet is +EV when your estimated probability of winning exceeds the implied probability embedded in the odds. This notebook teaches you how to find those edges using SportsQuant.

In [ ]:
# Core imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sportsquant.core.betting.odds import Odds
from sportsquant.core.betting.engine import expected_value, kelly_fraction, decide_over_under, BetDecision
from sportsquant.core.betting.kelly import EdgeCalculator, EdgeCalculatorConfig

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Imports successful. Let's find some +EV bets!")

## 1. What is Expected Value?

Expected Value tells you how much you *expect* to profit (or lose) per dollar wagered, on average, over many bets.

$$EV = P(\text{win}) \times (\text{odds} - 1) - P(\text{lose}) \times 1$$

- **+EV**: Your model says the event is more likely than the odds imply — you have an edge.
- **-EV**: The odds already overestimate the probability — you should pass.
- **Zero EV**: Break-even over the long run.

## 2. Converting Odds Formats

SportsQuant's `Odds` class handles American and decimal odds with conversion and implied probability.

In [ ]:
# Convert American odds to decimal and implied probability
odds_examples = [
    Odds(american=-110),   # Standard vig
    Odds(american=-105),   # Reduced vig
    Odds(american=+100),   # Even money
    Odds(american=+150),   # Underdog
    Odds(american=+200),   # Long shot
]

odds_df = pd.DataFrame({
    "American": [o.american for o in odds_examples],
    "Decimal": [o.to_decimal() for o in odds_examples],
    "Implied Prob": [o.implied_prob() for o in odds_examples],
})

odds_df["Vig"] = (1 - odds_df["Implied Prob"]) * 100  # approximate vig per side
odds_df.style.format({"Decimal": "{:.3f}", "Implied Prob": "{:.3f}", "Vig": "{:.1f}%"})

## 3. Sample Odds Data

We create a realistic set of NBA player prop lines inline. Each row represents a betting opportunity with both the bookmaker's line/odds and *our* estimated true probability.

In [ ]:
np.random.seed(42)

# Sample NBA prop lines with our model's estimated probability
sample_lines = pd.DataFrame({
    "player": ["LeBron James", "Stephen Curry", "Luka Doncic", "Nikola Jokic", "Giannis Antetokounmpo",
               "Jayson Tatum", "Joel Embiid", "Shai Gilgeous-Alexander", "Kevin Durant", "Anthony Davis",
               "Trae Young", "Damian Lillard", "Ja Morant", "Devin Booker", "Donovan Mitchell"],
    "stat": ["Points", "Points", "Points", "Assists", "Points",
             "Points", "Points", "Points", "Points", "Rebounds",
             "Assists", "Points", "Points", "Points", "Points"],
    "line": [27.5, 28.5, 32.5, 9.5, 31.5,
             28.5, 30.5, 29.5, 26.5, 11.5,
             9.5, 26.5, 24.5, 27.5, 25.5],
    "american_odds_over": [-110, -110, -115, -105, -110,
                           -110, -110, -115, -105, -110,
                           -105, -110, -110, -110, -110],
    "american_odds_under": [-110, -110, -105, -115, -110,
                            -110, -110, -105, -115, -110,
                            -115, -110, -110, -110, -110],
    "our_prob_over": [0.58, 0.53, 0.57, 0.56, 0.55,
                      0.48, 0.54, 0.56, 0.49, 0.52,
                      0.54, 0.46, 0.52, 0.47, 0.51],
})

sample_lines["implied_prob_over"] = sample_lines["american_odds_over"].apply(
    lambda x: Odds(american=x).implied_prob()
)

sample_lines.style.format({"our_prob_over": "{:.2f}", "implied_prob_over": "{:.3f}"})

## 4. Calculating Expected Value for Single Bets

Using `EdgeCalculator.compute_expected_value()` to find which props are +EV.

In [ ]:
edge_calc = EdgeCalculator()

results = []
for _, row in sample_lines.iterrows():
    odds_over = Odds(american=row["american_odds_over"])
    dec = odds_over.to_decimal()
    ev = edge_calc.compute_expected_value(row["our_prob_over"], dec, stake=1.0)
    edge = edge_calc.compute_edge(row["our_prob_over"], dec)
    results.append({
        "player": row["player"],
        "stat": row["stat"],
        "line": row["line"],
        "our_prob": row["our_prob_over"],
        "implied_prob": row["implied_prob_over"],
        "edge": edge,
        "ev_per_dollar": ev,
    })

ev_df = pd.DataFrame(results).sort_values("ev_per_dollar", ascending=False)

# Highlight +EV bets
ev_df["is_positive_ev"] = ev_df["ev_per_dollar"] > 0
ev_df.style.format({"our_prob": "{:.2f}", "implied_prob": "{:.3f}", "edge": "{:+.3f}", "ev_per_dollar": "{:+.4f}"})
  .applymap(lambda v: "background-color: #d4edda" if v else "", subset=["is_positive_ev"])

## 5. How Edge Changes with Different Odds at the Same Line

The same probability estimate yields different edges depending on the odds. Better odds (more favorable vig) mean more +EV.

In [ ]:
# Fix probability at 0.57, vary odds from -120 to +120
odds_range = list(range(-120, 125, 5))
true_prob = 0.57

edges = []
evs = []
for amer in odds_range:
    if amer == 0:
        continue
    o = Odds(american=amer)
    dec = o.to_decimal()
    edges.append(edge_calc.compute_edge(true_prob, dec))
    evs.append(edge_calc.compute_expected_value(true_prob, dec, stake=1.0))

fig, ax1 = plt.subplots(figsize=(12, 6))
valid_odds = [o for o in odds_range if o != 0]

ax1.bar(valid_odds, [e * 100 for e in edges], color=["green" if e > 0 else "red" for e in edges], alpha=0.6, width=4)
ax1.axhline(y=0, color="black", linestyle="-", linewidth=1)
ax1.set_xlabel("American Odds")
ax1.set_ylabel("Edge (%)", color="green")
ax1.set_title(f"Edge vs American Odds (True Probability = {true_prob:.0%})")
ax1.tick_params(axis="y", labelcolor="green")

ax2 = ax1.twinx()
ax2.plot(valid_odds, evs, color="blue", linewidth=2, label="EV per $1")
ax2.set_ylabel("EV per $1 Staked", color="blue")
ax2.tick_params(axis="y", labelcolor="blue")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

## 6. Automatic Recommendations with `decide_over_under()`

SportsQuant's `decide_over_under()` evaluates both sides and returns a `BetDecision` with the recommended side, EV, and Kelly fraction.

In [ ]:
decisions = []
for _, row in sample_lines.iterrows():
    odds_over = Odds(american=row["american_odds_over"])
    odds_under = Odds(american=row["american_odds_under"])
    p_over = row["our_prob_over"]
    
    decision = decide_over_under(
        line=row["line"],
        p_over=p_over,
        odds_over=odds_over,
        odds_under=odds_under,
        true_prob_over=p_over,
        true_prob_under=1.0 - p_over,
    )
    
    decisions.append({
        "player": row["player"],
        "stat": row["stat"],
        "line": row["line"],
        "side": decision.side,
        "p_win": decision.p_win,
        "ev": decision.ev,
        "kelly_fraction": decision.kelly_fraction,
        "decimal_odds": decision.decimal_odds,
    })

dec_df = pd.DataFrame(decisions).sort_values("ev", ascending=False)
dec_df["+EV"] = dec_df["ev"].apply(lambda x: "YES" if x > 0 else "NO")
dec_df.style.format({"p_win": "{:.3f}", "ev": "{:+.4f}", "kelly_fraction": "{:.4f}", "decimal_odds": "{:.3f}"})

## 7. EV vs Probability Curve

The visualization below shows the relationship between your estimated probability and EV for a given set of odds. The **breakeven probability** is where EV crosses zero.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel 1: EV curves for different odds
prob_range = np.linspace(0.3, 0.8, 200)
odds_configs = [
    (Odds(american=-110), "-110 (standard)"),
    (Odds(american=-105), "-105 (reduced vig)"),
    (Odds(american=+100), "+100 (even money)"),
    (Odds(american=+110), "+110 (positive EV odds)"),
]

for odds_obj, label in odds_configs:
    dec = odds_obj.to_decimal()
    ev_curve = [edge_calc.compute_expected_value(p, dec, stake=1.0) for p in prob_range]
    breakeven = odds_obj.implied_prob()
    axes[0].plot(prob_range, ev_curve, label=f"{label} (breakeven={breakeven:.2f})", linewidth=2)

axes[0].axhline(y=0, color="black", linestyle="--", alpha=0.5)
axes[0].set_xlabel("Your Estimated Probability")
axes[0].set_ylabel("Expected Value per $1")
axes[0].set_title("EV vs Your Probability Estimate")
axes[0].legend(fontsize=9)

# Panel 2: Edge distribution from sample data
positive_ev = ev_df[ev_df["ev_per_dollar"] > 0]
negative_ev = ev_df[ev_df["ev_per_dollar"] <= 0]

axes[1].barh(positive_ev["player"], positive_ev["ev_per_dollar"], color="green", alpha=0.7, label="+EV")
axes[1].barh(negative_ev["player"], negative_ev["ev_per_dollar"], color="red", alpha=0.7, label="-EV")
axes[1].axvline(x=0, color="black", linewidth=1)
axes[1].set_xlabel("EV per $1 Staked")
axes[1].set_title("EV for Each Prop (Sorted by EV)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Statistical Significance: Confidence Intervals

Just because a bet is +EV doesn't mean the edge is *real*. The `EdgeCalculator` provides Wilson score confidence intervals to test statistical significance.

In [ ]:
sig_results = []
for _, row in sample_lines.iterrows():
    odds_over = Odds(american=row["american_odds_over"])
    dec = odds_over.to_decimal()
    implied = odds_over.implied_prob()
    
    # Simulate different sample sizes (games observed)
    for n_obs in [20, 50, 100, 200]:
        lower, upper, margin = edge_calc.confidence_interval(
            row["our_prob_over"], n_obs, confidence=0.95
        )
        is_sig = edge_calc.is_significant_edge(
            row["our_prob_over"], dec, n_obs, confidence=0.95
        )
        sig_results.append({
            "player": row["player"],
            "our_prob": row["our_prob_over"],
            "implied": implied,
            "n_obs": n_obs,
            "ci_lower": lower,
            "ci_upper": upper,
            "margin_of_error": margin,
            "significant": is_sig,
        })

sig_df = pd.DataFrame(sig_results)
# Show a few +EV props with significance
pos_ev_players = ev_df[ev_df["ev_per_dollar"] > 0]["player"].tolist()
sig_df[sig_df["player"].isin(pos_ev_players)].style.format({
    "our_prob": "{:.2f}", "implied": "{:.3f}",
    "ci_lower": "{:.3f}", "ci_upper": "{:.3f}", "margin_of_error": "{:.3f}"
})

In [ ]:
# Visualize how CI narrows with more data for LeBron
lebron_sig = sig_df[sig_df["player"] == "LeBron James"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.errorbar(
    lebron_sig["n_obs"],
    lebron_sig["our_prob"],
    yerr=lebron_sig["margin_of_error"],
    fmt="o-",
    capsize=5,
    linewidth=2,
    color="blue",
    label="95% CI",
)
ax.axhline(y=Odds(american=-110).implied_prob(), color="red", linestyle="--", label="Implied prob (-110)")
ax.set_xlabel("Number of Observations")
ax.set_ylabel("Probability Estimate")
ax.set_title("Confidence Interval Narrows with More Data (LeBron James Over 27.5 Pts)")
ax.legend()
plt.tight_layout()
plt.show()

## Key Takeaway

> **A bet is +EV when your estimated probability exceeds the implied probability.**

1. Always convert odds to implied probability using `Odds.implied_prob()`
2. Calculate edge: `your_prob - implied_prob`
3. Use `decide_over_under()` for automatic over/under recommendations
4. Verify significance with `is_significant_edge()` — a +EV bet with a wide CI may not be reliable
5. Only bet when you have a genuine, statistically significant edge